In [5]:
from docx import Document
from pathlib import Path
import json
def extract_images_from_part(part, output_dir, manifest, start_index):
    image_index = start_index

    for rel_id, rel in part.rels.items():
        if "image" not in rel.reltype:
            continue

        image_index += 1
        image_part = rel.target_part
        image_bytes = image_part.blob

        content_type = image_part.content_type
        ext = content_type.split("/")[-1]

        image_name = f"image_{image_index:02d}.{ext}"
        image_path = output_dir / image_name

        with open(image_path, "wb") as f:
            f.write(image_bytes)

        manifest.append({
            "image_id": f"image_{image_index:02d}",
            "filename": image_name,
            "rel_id": rel_id,
            "content_type": content_type,
            "source_part": part.partname
        })

    return image_index

def extract_all_images_from_docx(docx_path, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    doc = Document(docx_path)
    manifest = []
    image_index = 0

    # 1️⃣ Main document body
    image_index = extract_images_from_part(
        doc.part, output_dir, manifest, image_index
    )

    # 2️⃣ Headers and footers (THIS FIXES MANY "MISSING" CASES)
    for section in doc.sections:
        if section.header:
            image_index = extract_images_from_part(
                section.header.part, output_dir, manifest, image_index
            )

        if section.footer:
            image_index = extract_images_from_part(
                section.footer.part, output_dir, manifest, image_index
            )

    # Save manifest
    manifest_path = output_dir / "manifest.json"
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    return manifest


In [6]:
file = r"\\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\Final\BTS_Port-Performance-2026_Annual-Report_DRAFT for BTS_12.5.25_asof_12.10.docx"
output_directory = "extracted_images"
extract_all_images_from_docx(file, output_directory)    


[{'image_id': 'image_01',
  'filename': 'image_01.png',
  'rel_id': 'rId68',
  'content_type': 'image/png',
  'source_part': '/word/document.xml'},
 {'image_id': 'image_02',
  'filename': 'image_02.png',
  'rel_id': 'rId84',
  'content_type': 'image/png',
  'source_part': '/word/document.xml'},
 {'image_id': 'image_03',
  'filename': 'image_03.png',
  'rel_id': 'rId16',
  'content_type': 'image/png',
  'source_part': '/word/document.xml'},
 {'image_id': 'image_04',
  'filename': 'image_04.png',
  'rel_id': 'rId11',
  'content_type': 'image/png',
  'source_part': '/word/document.xml'},
 {'image_id': 'image_05',
  'filename': 'image_05.png',
  'rel_id': 'rId37',
  'content_type': 'image/png',
  'source_part': '/word/document.xml'},
 {'image_id': 'image_06',
  'filename': 'image_06.png',
  'rel_id': 'rId90',
  'content_type': 'image/png',
  'source_part': '/word/document.xml'},
 {'image_id': 'image_07',
  'filename': 'image_07.png',
  'rel_id': 'rId22',
  'content_type': 'image/png',
  's